# 43 — Drift & calibration monitoring

Feature/score drift and high-score-tail calibration. TEST 2025 appears only as a **consumed historical benchmark**, never as a selection set.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

assert (paths.reports / "robustness" / "42_validation.parquet").exists(), "Ejecuta 42 primero."
from argos.robustness.drift import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-43-feature-score-calibration-drift-v1'

In [2]:
from argos.robustness.drift import build_drift_monitoring

result = build_drift_monitoring(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.feature_drift.sort_values(["status", "drift_value"], ascending=[True, False]).head(30)


,feature,family,reference_period,monitor_period,metric,drift_value,unseen_category_rate,status
69,entity_tenure_days,numeric,2023,2025,psi,1.155082,NaN,high
28,entity_tenure_days,numeric,2023,2024,psi,0.693451,NaN,high
67,supplier_tenure_days,numeric,2023,2025,psi,0.260876,NaN,high
10,codigo_de_categoria_principal,categorical,2023,2024,total_variation_distance,0.090070,0.01417,stable
68,days_since_supplier_last_contract,numeric,2023,2025,psi,0.086794,NaN,stable
25,prior_supplier_entity_contract_count,numeric,2023,2024,psi,0.085047,NaN,stable
5,ciudad,categorical,2023,2024,total_variation_distance,0.081230,0.00714,stable
24,prior_entity_contract_count,numeric,2023,2024,psi,0.077777,NaN,stable
66,prior_supplier_entity_contract_count,numeric,2023,2025,psi,0.073652,NaN,stable
23,prior_supplier_contract_count,numeric,2023,2024,psi,0.067465,NaN,stable


In [3]:
result.score_drift


,period,rows,prevalence,score_mean,score_std,score_p50,score_p90,score_p99,score_p999,score_psi_vs_validation_h2,usage
0,validation_2024_h2,428076,0.003663,0.006003,0.013038,0.002712,0.007437,0.071359,0.130938,0.000000,development_reference
1,test_2025_consumed_benchmark,1050857,0.010451,0.010553,0.019440,0.003360,0.021159,0.130938,0.130938,0.326331,historical_benchmark_only_not_selection


In [4]:
result.tail_calibration


,period,tail_bin,rows,positives,mean_predicted_probability,observed_rate,min_score,max_score,calibration_gap
0,validation_2024_h2,bottom_90,385268,519,0.003091,0.001347,1.000000e-07,0.007437,-0.001744
1,validation_2024_h2,90_95,21404,240,0.014096,0.011213,7.437287e-03,0.019697,-0.002883
2,validation_2024_h2,95_99,17123,548,0.035075,0.032004,1.969669e-02,0.071359,-0.003072
3,validation_2024_h2,99_99_5,2140,155,0.091597,0.072430,7.135929e-02,0.130938,-0.019167
4,validation_2024_h2,99_5_99_9,1712,84,0.130938,0.049065,1.309385e-01,0.130938,-0.081873
5,validation_2024_h2,top_0_1,429,22,0.130938,0.051282,1.309385e-01,0.130938,-0.079656
6,test_2025_consumed_benchmark,bottom_90,945771,3846,0.005311,0.004067,1.000000e-07,0.021159,-0.001245
7,test_2025_consumed_benchmark,90_95,52543,1367,0.034498,0.026017,2.115915e-02,0.047740,-0.008481
8,test_2025_consumed_benchmark,95_99,42034,2983,0.068455,0.070966,4.774005e-02,0.130938,0.002512
9,test_2025_consumed_benchmark,99_99_5,5254,1368,0.130938,0.260373,1.309385e-01,0.130938,0.129435


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,drift_feature_coverage,drift,True,True,"{""features"": 41, ""periods"": [""2024"", ""2025""]}",every frozen V1 feature is monitored across th...
1,drift_values_finite,quality,True,True,"{""nonfinite"": 0}",feature drift metrics are finite and auditable
2,consumed_test_is_diagnostic_only,governance,True,True,"{""test_usage"": [""historical_benchmark_only_not...",TEST 2025 may be monitored post hoc but is nev...
3,tail_calibration_persisted,calibration,True,True,"{""periods"": [""validation_2024_h2"", ""test_2025_...",calibration is monitored explicitly in the hig...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("43 ✅ PASS" if result.ready_for_next_notebook else "43 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


43 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_drift_monitoring.json
